# 🦋 Insect Anatomy YOLO-Seg (v2: High-Res + Data Augmentation)

This notebook trains a higher-capacity YOLO instance segmentation model on your insect annotations.

### What's New in v2:
1. **Bigger Model (`yolov8s-seg` / `yolo11s-seg`):** ~11M parameters (3.5x capacity of nano) to capture complex wing veins, pectinate antennae, and legs on textured backgrounds.
2. **Higher Resolution (`imgsz=1024`):** Retains delicate sub-millimeter appendages that get lost at 640px.
3. **Domain Adaptation Augmentations:** Aggressive HSV lighting jitter (sunlight vs shade vs flash), scale variations (+/- 50% for macro vs full insect), random rotation (+/- 30°), and Mosaic/Mixup.
4. **Verified iNat Samples Mixed In:** Learns directly from real iNaturalist outdoor photos.

### 1. Check Hardware & Sync Latest Annotations

In [ ]:
import torch
import os

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🚀 PyTorch version: {torch.__version__}")
print(f"⚡ Compute device:  {device} ({'GPU: ' + torch.cuda.get_device_name(0) if device != 'cpu' else 'CPU multi-core'})")

# Sync any recently verified iNat samples into the training dataset
!python cluster_yolo_training/update_dataset.py

### 2. Verify Dataset
Checks images and classes in `dataset.yaml`.

In [ ]:
import glob
import os
import yaml

dataset_dir = os.path.abspath('cluster_yolo_training/dataset' if os.path.exists('cluster_yolo_training/dataset') else 'dataset')
yaml_path = os.path.join(dataset_dir, 'dataset.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(f'''path: {dataset_dir}
train: images/train
val: images/val

names:
  0: full-body
  1: wing
  2: leg
  3: antenna
  4: head
  5: body
''')

with open(yaml_path) as f:
    dataset_cfg = yaml.safe_load(f)

print("Classes:", dataset_cfg.get('names'))
train_imgs = glob.glob(os.path.join(dataset_dir, 'images/train/*.jpg'))
val_imgs = glob.glob(os.path.join(dataset_dir, 'images/val/*.jpg'))
print(f"Train samples: {len(train_imgs)} (includes colleague field + downscaled iNat!)")
print(f"Val samples:   {len(val_imgs)}")

### 2b. Preview Data Augmentations (Field & Macro Domain Adaptations)
Visualizes simulated training augmentations (HSV lighting/flash shifts, rotation, zoom, blur, and sensor grain) on both a field camera trap image and an iNaturalist sample.
If a training run has already started, this also displays YOLO's native training batch mosaic.

In [ ]:
import os
import glob
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

# Robust dataset discovery across all Jupyter launch directories
candidate_paths = [
    "cluster_yolo_training/dataset",
    "dataset",
    "antenna/cluster_yolo_training/dataset",
    "../cluster_yolo_training/dataset",
    os.path.expanduser("~/antenna/cluster_yolo_training/dataset"),
    os.path.expanduser("~/antenna/antenna/cluster_yolo_training/dataset")
]

dataset_dir = None
for p in candidate_paths:
    if os.path.isdir(os.path.join(p, "images", "train")):
        train_test = glob.glob(os.path.join(p, "images", "train", "*.jpg"))
        if len(train_test) > 0:
            dataset_dir = os.path.abspath(p)
            break

if dataset_dir is None:
    print("❌ ERROR: Could not find dataset directory!")
    print(f"Current working directory (CWD): {os.getcwd()}")
    print("Searched locations:", candidate_paths)
    print("If you cloned inside another antenna folder, check if you have a nested antenna/antenna/ directory.")
else:
    train_imgs = glob.glob(os.path.join(dataset_dir, "images/train/*.jpg"))
    print(f"✔ Found {len(train_imgs)} training images in: {dataset_dir}")

    field_imgs = [f for f in train_imgs if "ind_" in os.path.basename(f)]
    inat_imgs = [f for f in train_imgs if "ind_" not in os.path.basename(f)]

    samples_to_show = []
    if field_imgs:
        samples_to_show.append(("Field Camera Trap", random.choice(field_imgs)))
    if inat_imgs:
        samples_to_show.append(("Degraded iNaturalist", random.choice(inat_imgs)))
    if not samples_to_show and train_imgs:
        samples_to_show.append(("Sample Image", random.choice(train_imgs)))

    if len(samples_to_show) == 0:
        print("⚠️ No images found to preview augmentations.")
    else:
        def apply_simulated_augs(bgr):
            h, w = bgr.shape[:2]
            # 1. Lighting / HSV shift (flash / low light)
            hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
            hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-10, 10)) % 180
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.6, 1.4), 0, 255)
            hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.6, 1.4), 0, 255)
            img_hsv = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

            # 2. Rotation & Zoom
            angle = random.uniform(-30, 30)
            scale = random.uniform(0.7, 1.3)
            M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
            img_geom = cv2.warpAffine(bgr, M, (w, h), borderMode=cv2.BORDER_REFLECT)

            # 3. Field blur + sensor grain
            ksize = random.choice([3, 5])
            img_blur = cv2.GaussianBlur(img_geom, (ksize, ksize), 0)
            noise = np.random.normal(0, random.uniform(6, 14), bgr.shape).astype(np.float32)
            img_combined = np.clip(img_blur.astype(np.float32) + noise, 0, 255).astype(np.uint8)

            return img_hsv, img_geom, img_combined

        # Plot augmentations grid
        fig, axes = plt.subplots(len(samples_to_show), 4, figsize=(16, 4 * len(samples_to_show)))
        if len(samples_to_show) == 1:
            axes = np.expand_dims(axes, 0)

        col_titles = [
            "1. Original Dataset Image",
            "2. HSV Lighting Shift (Flash/Shadow)",
            "3. Rotation (±30°) & Zoom (±50%)",
            "4. Combined (Rotation + Noise + Blur)"
        ]

        for row_idx, (tag, img_path) in enumerate(samples_to_show):
            bgr = cv2.imread(img_path)
            aug1, aug2, aug3 = apply_simulated_augs(bgr)
            
            imgs = [bgr, aug1, aug2, aug3]
            for col_idx, im in enumerate(imgs):
                ax = axes[row_idx, col_idx]
                rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
                ax.imshow(rgb)
                ax.set_title(f"{tag} - {col_titles[col_idx]}", fontsize=10)
                ax.axis("off")

        plt.tight_layout()
        plt.show()

        # If training has already run, show native YOLO mosaic batch
        batch_mosaic = "runs/insect_anatomy_v2/train_batch0.jpg"
        if os.path.exists(batch_mosaic):
            print("🎨 Native YOLO Multi-Image Training Mosaic:")
            display(Image(batch_mosaic, width=700))


### 3. Start Training (v2 with High-Res & Augmentations)
- Model: `yolov8s-seg.pt` (Small)
- Resolution: `1024` (preserves fine antenna and leg textures)
- Epochs: `150` with Cosine Annealing learning rate

In [ ]:
from ultralytics import YOLO

# Load small model with 3.5x capacity of nano
model = YOLO('yolov8s-seg.pt')

batch_size = 16 if torch.cuda.is_available() else 4
epochs = 150

results = model.train(
    data=yaml_path,
    epochs=epochs,
    batch=batch_size,
    imgsz=640,
    device=device,
    patience=30,
    cos_lr=True,            # Cosine learning rate schedule
    # --- Domain Augmentations for iNat & Macro Lighting ---
    hsv_h=0.015,            # Hue jitter
    hsv_s=0.7,              # Saturation jitter (sun vs shade)
    hsv_v=0.4,              # Value / brightness jitter (shadows vs flash)
    degrees=30.0,           # Random rotations up to 30 deg
    scale=0.5,              # Multi-scale zoom (+/- 50%)
    fliplr=0.5,             # Horizontal flips
    flipud=0.5,             # Vertical flips
    mosaic=1.0,             # Multi-image context mosaic
    mixup=0.15,             # Blend overlapping structures
    close_mosaic=15,        # Turn off mosaic for clean boundary learning
    project='runs',
    name='insect_anatomy_v2',
    exist_ok=True,
    save=True,
    plots=True
)

### 4. Inspect Results & Loss Curves

In [ ]:
from IPython.display import Image, display

results_png = 'runs/insect_anatomy_v2/results.png'
if os.path.exists(results_png):
    display(Image(results_png))
else:
    print("Results plot not found yet.")

### 4b. Visual Segmentation Results (Field & iNat Test Set)
Runs inference on validation images using  and displays predicted segmentation masks, bounding boxes, and anatomical part counts inline.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import glob, os, random, cv2
from collections import Counter

os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

# 1. Load trained best weights
weights_candidates = [
    "cluster_yolo_training/weights/best.pt",
    "runs/insect_anatomy_v2/weights/best.pt",
    "weights/best.pt"
]
best_weights = next((w for w in weights_candidates if os.path.exists(w)), None)
assert best_weights is not None, "❌ Trained weights (best.pt) not found. Run training first!"
print(f"✔ Loading model from: {best_weights}")
val_model = YOLO(best_weights)

# 2. Select a representative sample of field camera trap + iNat validation images
val_dir = "dataset/images/val" if os.path.exists("dataset/images/val") else "cluster_yolo_training/dataset/images/val"
val_all = sorted(glob.glob(os.path.join(val_dir, "*.jpg")))

val_field = [f for f in val_all if "ind_" in os.path.basename(f)]
val_inat = [f for f in val_all if "ind_" not in os.path.basename(f)]

# Pick 4 field images and 2 iNat images (or whatever is available)
selected = []
if val_field:
    selected.extend(val_field[:4])
if val_inat:
    selected.extend(val_inat[:2])
if not selected:
    selected = val_all[:6]

print(f"🔍 Running inference on {len(selected)} validation images (Conf >= 0.25)...")
results = val_model.predict(source=selected, conf=0.25, iou=0.45, max_det=30, verbose=False)

# 3. Display segmentation results directly in notebook
num_imgs = len(results)
cols = 3
rows = (num_imgs + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for idx, (r, ax) in enumerate(zip(results, axes)):
    # r.plot() draws colored polygon masks, boxes, labels & confidences
    annotated_rgb = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    
    # Count detected classes
    counts = Counter()
    if r.boxes and r.boxes.cls is not None:
        for c in r.boxes.cls.cpu().numpy():
            cls_name = r.names.get(int(c), str(int(c)))
            counts[cls_name] += 1
    
    parts_summary = ", ".join(f"{cnt} {cls}" for cls, cnt in counts.items()) if counts else "No detections"
    fname = os.path.basename(r.path)
    source_tag = "📷 FIELD" if "ind_" in fname else "🌿 iNAT"
    
    ax.set_title(title_str, fontsize=11, fontweight="bold")
    ax.axis("off")

# Hide any extra empty subplots
for j in range(idx + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


### 4c. Comparative Benchmark: Pure YOLO vs. YOLO + SAM
Directly evaluates how **YOLO bounding boxes + Segment Anything (SAM)** compare against **Pure YOLO** on the validation set.
Generates side-by-side 4-panel figures: `[Original]` | `[Ground Truth]` | `[Pure YOLO]` | `[YOLO + SAM]`

In [ ]:
import os, glob
from IPython.display import Image, display

# 1. Run the comparative benchmark script
!python cluster_yolo_training/compare_yolo_sam.py --num-samples 6

# 2. Display comparison panels inline
cmp_plots = sorted(glob.glob('cluster_yolo_training/runs/yolo_vs_sam/*.jpg'))
print(f'\n🖼️ Displaying {min(4, len(cmp_plots))} sample comparisons:')
for p in cmp_plots[:4]:
    display(Image(p, width=950))


### 5. Save & Export Weights

In [ ]:
import shutil

os.makedirs('cluster_yolo_training/weights', exist_ok=True)
best_src = 'runs/insect_anatomy_v2/weights/best.pt'
best_dst = 'cluster_yolo_training/weights/best.pt'

if os.path.exists(best_src):
    shutil.copy2(best_src, best_dst)
    print(f"✔ Saved best model to: {best_dst}")
    
    best_model = YOLO(best_dst)
    onnx_path = best_model.export(format='onnx', imgsz=1024)
    print(f"✔ Exported to ONNX: {onnx_path}")

### 6. Auto-Annotate Active Backlog with Trained Model
Loads your trained model and generates LabelMe polygon masks for unannotated images in `data/active_annotation_set`.

> **🛡️ STRICT SAFETY GUARANTEE:** Any image or shape created, verified (`checked=True`), or edited by a human is **100% protected** and **never overwritten**!


In [2]:
import os
import glob
import json
from collections import defaultdict
from PIL import Image
from ultralytics import YOLO

# ==============================================================================
# 1. LOAD MODEL WEIGHTS
# ==============================================================================
weights_candidates = [
    'runs/insect_anatomy_v2/weights/best.pt'
]
# Or manually specify: weights_path = 'path/to/my_weights.pt'
weights_path = next((w for w in weights_candidates if os.path.exists(w)), None)
assert weights_path is not None, f'❌ Model weights not found in {weights_candidates}'
print(f'🤖 Loading trained model: {weights_path}')
annot_model = YOLO(weights_path)

# ==============================================================================
# 2. LOCATE ACTIVE ANNOTATION TARGET DIRECTORY
# ==============================================================================
target_candidates = [
    'data/active_annotation_set',
    '../data/active_annotation_set',
    os.path.expanduser('~/antenna/data/active_annotation_set'),
    os.path.expanduser('~/antenna/antenna/data/active_annotation_set')
]
target_dir = next((t for t in target_candidates if os.path.exists(t)), None)
assert target_dir is not None, f'❌ Target directory not found in {target_candidates}'
print(f'📁 Target backlog folder: {target_dir}')

# ==============================================================================
# 3. SETTINGS & SAFETY RULES
# ==============================================================================
CONF_THRESH = 0.25      # Detection confidence threshold
IOU_THRESH = 0.45       # NMS IoU threshold
TRACK_FILTER = 'all'    # 'all', 'field', or 'inat'
OVERWRITE_AI = True     # True = refresh AI masks with new model.
                        # Real/manual annotations are ALWAYS strictly preserved!

def is_protected_annotation(json_path):
    '''
    CRITICAL SAFETY CHECK:
    Returns True if an annotation has been created, verified, or edited by a human.
    These are 100% PROTECTED and NEVER overwritten!
    '''
    if not os.path.exists(json_path):
        return False
    try:
        with open(json_path, 'r', encoding='utf-8') as fp:
            d = json.load(fp)
        # 1. Check if marked verified or checked in AnyLabeling
        if d.get('checked') is True:
            return True
        flags = d.get('flags', {})
        if flags and flags.get('status') == 'verified':
            return True
        # 2. Check if pure manual annotation (no ai_generated flag)
        if not flags or not flags.get('ai_generated'):
            return True
        # 3. Check if human added or modified any polygon shape
        shapes = d.get('shapes', [])
        if any(not s.get('description', '').startswith('🤖 AI') for s in shapes):
            return True
        return False
    except Exception:
        return True  # Erring on the side of caution

# ==============================================================================
# 4. RUN INFERENCE & AUTO-ANNOTATE
# ==============================================================================
search_dirs = []
if TRACK_FILTER in ['all', 'field']:
    search_dirs.append(os.path.join(target_dir, 'field'))
if TRACK_FILTER in ['all', 'inat']:
    search_dirs.append(os.path.join(target_dir, 'inat'))

img_files = []
for sdir in search_dirs:
    if os.path.exists(sdir):
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
            img_files.extend(glob.glob(os.path.join(sdir, '**', ext), recursive=True))

img_files = sorted(set(img_files))
print(f'🔍 Found {len(img_files)} candidate images to process...\n')

annotated_count = 0
skipped_manual = 0
skipped_existing_ai = 0
part_stats = defaultdict(lambda: defaultdict(int))

for idx, img_path in enumerate(img_files, 1):
    json_path = os.path.splitext(img_path)[0] + '.json'

    # STRICT SAFETY: Skip all manual / human-verified files
    if os.path.exists(json_path):
        if is_protected_annotation(json_path):
            skipped_manual += 1
            continue
        elif not OVERWRITE_AI:
            skipped_existing_ai += 1
            continue

    track = 'field' if 'field' in img_path else 'inat'

    with Image.open(img_path) as im:
        width, height = im.size

    results = annot_model.predict(source=img_path, conf=CONF_THRESH, iou=IOU_THRESH, max_det=30, verbose=False)
    result = results[0]

    shapes = []
    if result.masks is not None:
        polygons = result.masks.xy
        classes = result.boxes.cls.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()

        for poly, cls_id, score in zip(polygons, classes, confs):
            if len(poly) < 3:
                continue
            label_name = result.names[int(cls_id)]
            part_stats[track][label_name] += 1
            points = [[round(float(pt[0]), 2), round(float(pt[1]), 2)] for pt in poly]

            shapes.append({
                'label': label_name,
                'score': round(float(score), 4),
                'points': points,
                'group_id': None,
                'description': f'🤖 AI ({label_name} {score:.0%})',
                'difficult': False,
                'shape_type': 'polygon',
                'flags': {'ai_predicted': True},
                'attributes': {},
                'kie_linking': []
            })

    labelme_data = {
        'version': '5.0.1',
        'flags': {
            'ai_generated': True,
            'model': os.path.basename(weights_path),
            'status': 'needs_human_review'
        },
        'checked': False,
        'shapes': shapes,
        'imagePath': os.path.basename(img_path),
        'imageData': None,
        'imageHeight': height,
        'imageWidth': width
    }

    with open(json_path, 'w', encoding='utf-8') as fp:
        json.dump(labelme_data, fp, indent=2)

    annotated_count += 1
    if idx % 50 == 0 or idx == len(img_files):
        print(f'  [{idx}/{len(img_files)}] Processed ({annotated_count} annotated, {skipped_manual} manual preserved)')

print('\n' + '=' * 70)
print('📊 PRE-ANNOTATION COMPLETE')
print('=' * 70)
print(f'🛡️  HUMAN/VERIFIED ANNOTATIONS PRESERVED: {skipped_manual} (100% untouched)')
print(f'🤖  AI ANNOTATIONS GENERATED / UPDATED:   {annotated_count}')
print(f'⏭️  EXISTING AI SKIPPED:                 {skipped_existing_ai}')
print('-' * 70)
print('DETECTED ANATOMICAL PARTS:')
for trk, parts in part_stats.items():
    print(f'  • {trk.upper():<6} -> ' + ', '.join(f'{k}: {v}' for k, v in sorted(parts.items())))
print('=' * 70)
print('✔ Complete! Open AnyLabeling to review and approve masks.')


🤖 Loading trained model: runs/insect_anatomy_v2/weights/best.pt
📁 Target backlog folder: ../data/active_annotation_set
🔍 Found 722 candidate images to process...

  [50/722] Processed (50 annotated, 0 manual preserved)
  [100/722] Processed (100 annotated, 0 manual preserved)
  [150/722] Processed (150 annotated, 0 manual preserved)
  [200/722] Processed (200 annotated, 0 manual preserved)
  [250/722] Processed (250 annotated, 0 manual preserved)
  [300/722] Processed (300 annotated, 0 manual preserved)
  [350/722] Processed (350 annotated, 0 manual preserved)
  [400/722] Processed (400 annotated, 0 manual preserved)
  [450/722] Processed (437 annotated, 13 manual preserved)
  [500/722] Processed (473 annotated, 27 manual preserved)
  [550/722] Processed (514 annotated, 36 manual preserved)
  [600/722] Processed (557 annotated, 43 manual preserved)
  [650/722] Processed (595 annotated, 55 manual preserved)

📊 PRE-ANNOTATION COMPLETE
🛡️  HUMAN/VERIFIED ANNOTATIONS PRESERVED: 100 (100% u